# Mortalidade por Câncer de Mama — Análise Exploratória de Dados

**Projeto:** TCC — Análise de Dados de Câncer de Mama em Municípios de Minas Gerais  
**Metodologia:** KDD (Knowledge Discovery in Databases)  
**Etapa:** Análise Exploratória de Dados (EDA)  
**Escopo:** Mortalidade por faixa etária (1979–2024) — municípios e regionais de saúde de MG

---
## 1. Imports

In [ ]:
import json
import re
import unicodedata
from pathlib import Path

import geobr
import geopandas as gpd
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import Markdown, display

---
## 2. Configurações Gerais

Centraliza todos os parâmetros configuráveis: caminhos, filtros, paletas de cores e constantes de layout.  
Para filtrar por município, regional ou código IBGE específico, ajuste as variáveis `MUNICIPIO`, `CODIGO_MUNICIPIO` ou `REGIONAL_SAUDE`. Deixe `None` para analisar o estado inteiro.

In [ ]:
# ── Filtros de escopo (None = sem filtro, analisa MG inteiro) ─────────────────
MUNICIPIO        = None
CODIGO_MUNICIPIO = None
REGIONAL_SAUDE   = None

# ── Caminhos dos arquivos de entrada ─────────────────────────────────────────
CSV_MORTALIDADE = "/home/anna/tcc/Tratamento/Mortalidade/tabela_mortalidade_faixa_etaria.csv"

# ── Período de análise ────────────────────────────────────────────────────────
ANOS_RANGE_MORTALIDADE  = (1979, 2024)
ANO_MALHA_MUNICIPIOS_MG = 2020         # ano de referência da malha geobr

ANO_INI_MAPA, ANO_FIM_MAPA = ANOS_RANGE_MORTALIDADE

# ── Faixas etárias padronizadas ───────────────────────────────────────────────
FAIXAS_ETARIAS_MORTALIDADE = [
    "0-9", "10-19", "20-29", "30-39",
    "40-49", "50-59", "60-69", "70 ou mais",
]

# ── Paleta de cores e layout dos gráficos ────────────────────────────────────
COR_BARRAS = "#6ca8e5"
COR_PONTOS = "#d73f3f"

LARGURA_GRAFICO            = 1900
ALTURA_GRAFICO_BARRAS      = 600
ALTURA_GRAFICO_DISPERSAO   = 900
X_MIN, X_MAX               = 0, 13000

FONTE_TITULO      = 28
FONTE_EIXO_TITULO = 20
FONTE_EIXO_TICK   = 18
FONTE_LEGENDA     = 22
FONTE_TEXTO_BARRA = 22
FONTE_TEXTO_PONTO = 16

TITULO_MORTALIDADE = "Mortalidade por faixa etária"

# ── Paleta e faixas para os mapas coropléticos ───────────────────────────────
COR_SEM_MORTALIDADE = "#6ca8e5"
PALETA_MAPA         = ["#9c2328", "#d20000", "#ff7f50", "#ee9c21", "#fec44f", "#ffffd4"]
FAIXAS_MUNICIPIOS   = [50, 100, 500, 1000, 5000]
FAIXAS_REGIONAIS    = [500, 1000, 1500, 2500, 5000]

---
## 3. Funções Auxiliares

Funções utilitárias reutilizadas ao longo de todo o notebook. Definidas uma única vez aqui para evitar repetição.

In [ ]:
# ── Normalização de texto e colunas ──────────────────────────────────────────

def _norm_txt(txt: str) -> str:
    """Remove acentos e converte para minúsculas (para comparações)."""
    return (
        unicodedata.normalize("NFKD", str(txt))
        .encode("ascii", "ignore")
        .decode("ascii")
        .lower()
        .strip()
    )


def _norm_col(col: str) -> str:
    """Normaliza nome de coluna: sem acento, minúsculo, espaços → underscore."""
    return re.sub(r"\s+", "_", _norm_txt(col))


def _find_first_col(df: pd.DataFrame, candidates_norm: set) -> str | None:
    """Retorna a primeira coluna do df cujo nome normalizado esteja em candidates_norm."""
    for col in df.columns:
        if _norm_col(col) in candidates_norm:
            return col
    return None


def _drop_index_cols(df: pd.DataFrame) -> pd.DataFrame:
    """Remove colunas 'Unnamed:...' geradas pelo pandas ao ler CSVs com índice salvo."""
    drop_cols = [c for c in df.columns if _norm_col(c).startswith("unnamed")]
    return df.drop(columns=drop_cols).copy() if drop_cols else df.copy()


def _norm_cod7(s: pd.Series) -> pd.Series:
    """Normaliza código IBGE para 7 dígitos com zero à esquerda."""
    x = pd.to_numeric(s, errors="coerce")
    return x.round().astype("Int64").astype(str).replace("<NA>", pd.NA).str.zfill(7)


# ── Helpers de cálculo ───────────────────────────────────────────────────────

def _fmt_int(n) -> str:
    """Formata inteiro com separador de milhar brasileiro (ponto)."""
    try:
        return f"{int(n):,}".replace(",", ".")
    except Exception:
        return str(n)


def _sum_row_wise(frame: pd.DataFrame, cols: list) -> pd.Series:
    """Soma colunas especificadas linha a linha, tratando nulos como zero."""
    if not cols:
        return pd.Series([0] * len(frame), index=frame.index, dtype="int64")
    return (
        frame[cols]
        .apply(pd.to_numeric, errors="coerce")
        .fillna(0)
        .sum(axis=1)
        .round()
        .astype(int)
    )


def _col_ano_faixa(ano: int, faixa: str) -> str:
    """Monta nome de coluna no padrão YYYY_faixa."""
    return f"{int(ano)}_{faixa}"


def _col_total_ano(ano: int) -> str:
    """Monta nome de coluna no padrão Total_YYYY."""
    return f"Total_{int(ano)}"


def _xticks_fixos(x_min: int, x_max: int, passo: int = 500) -> list:
    """Gera lista de ticks fixos para o eixo X."""
    return list(range(int(x_min), int(x_max) + 1, int(passo)))


# ── Helpers para mapas coropléticos ─────────────────────────────────────────

def _rotular_faixas_fixas(serie: pd.Series, faixas: list) -> tuple:
    """Classifica valores numéricos em faixas e retorna (série_de_rótulos, lista_de_labels_usados)."""
    s = pd.to_numeric(serie, errors="coerce").fillna(0)
    labels = ["Sem mortalidade (0)"]
    inicio = 1
    for i, limite in enumerate(faixas):
        if i == 0:
            labels.append(f"{inicio}–{limite}")
        else:
            labels.append(f"{faixas[i-1]+1:,}–{limite:,}".replace(",", "."))
        inicio = limite + 1
    labels.append(f"{faixas[-1]+1:,}+".replace(",", "."))
    classe = pd.Series("Sem mortalidade (0)", index=s.index)
    if (s > 0).any():
        inicio = 1
        for idx_faixa, limite in enumerate(faixas, start=1):
            classe.loc[(s >= inicio) & (s <= limite)] = labels[idx_faixa]
            inicio = limite + 1
        classe.loc[s > faixas[-1]] = labels[-1]
    labels_usados = ["Sem mortalidade (0)"] + [lab for lab in labels[1:] if (classe == lab).any()]
    return classe, labels_usados


def _categoria_reg(n: int, faixas: list) -> str:
    """Retorna o rótulo de faixa correspondente ao valor n (para regionais)."""
    labels = [
        f"0–{faixas[0]}",
        *[f"{faixas[i]+1:,}–{faixas[i+1]:,}".replace(",", ".") for i in range(len(faixas)-1)],
        f"{faixas[-1]+1:,}+".replace(",", "."),
    ]
    if n <= faixas[0]:
        return labels[0]
    for i in range(1, len(faixas)):
        if n <= faixas[i]:
            return labels[i]
    return labels[-1]


def _centro_gdf(gdf: gpd.GeoDataFrame) -> tuple:
    """Retorna (lat_centro, lon_centro) do bounding box de um GeoDataFrame."""
    b = gdf.total_bounds
    return (b[1] + b[3]) / 2, (b[0] + b[2]) / 2


print("✅ Funções auxiliares definidas.")

---
## 4. Carregamento e Filtragem dos Dados

Leitura do CSV de mortalidade e aplicação dos filtros configurados na seção 2 (município, código IBGE ou regional de saúde).

In [ ]:
def aplicar_filtros(df: pd.DataFrame) -> pd.DataFrame:
    """Aplica filtros de município, código IBGE e regional de saúde ao DataFrame."""
    out = _drop_index_cols(df)

    col_mun = _find_first_col(out, {"municipios", "municipio", "cidades", "cidade"})
    col_cod = _find_first_col(out, {"codigo", "cod"})
    col_reg = _find_first_col(out, {"regional_de_saude", "regional_saude", "regional"})

    if MUNICIPIO is not None:
        if not col_mun:
            raise KeyError("Não encontrei coluna de município para filtrar.")
        out = out[out[col_mun].astype(str).map(_norm_txt) == _norm_txt(MUNICIPIO)]

    if CODIGO_MUNICIPIO is not None:
        if not col_cod:
            raise KeyError("Não encontrei coluna de código para filtrar.")
        out = out[pd.to_numeric(out[col_cod], errors="coerce").astype("Int64") == int(CODIGO_MUNICIPIO)]

    if REGIONAL_SAUDE is not None:
        if not col_reg:
            raise KeyError("Não encontrei coluna de regional para filtrar.")
        out = out[out[col_reg].astype(str).map(_norm_txt) == _norm_txt(REGIONAL_SAUDE)]

    return out


# Leitura e filtragem
df_mort = pd.read_csv(CSV_MORTALIDADE)
f_mort  = aplicar_filtros(df_mort)

print("╔══════════════════════════════════════╗")
print("║       Configuração dos filtros       ║")
print("╠══════════════════════════════════════╣")
print(f"║  Município:        {str(MUNICIPIO):<20}║")
print(f"║  Código IBGE:      {str(CODIGO_MUNICIPIO):<20}║")
print(f"║  Regional:         {str(REGIONAL_SAUDE):<20}║")
print(f"║  Período:          {str(ANOS_RANGE_MORTALIDADE[0]) + '–' + str(ANOS_RANGE_MORTALIDADE[1]):<20}║")
print("╠══════════════════════════════════════╣")
print(f"║  Linhas no CSV:    {len(df_mort):<20}║")
print(f"║  Linhas filtradas: {len(f_mort):<20}║")
print("╚══════════════════════════════════════╝")

---
## 5. Análise Inicial dos Dados

Inspeção rápida da estrutura do DataFrame para verificar colunas, tipos e valores ausentes.

In [ ]:
display(Markdown(f"**Shape:** {f_mort.shape[0]} linhas × {f_mort.shape[1]} colunas"))

display(Markdown("**Primeiras linhas:**"))
display(f_mort.head(3))

missing = f_mort.isnull().sum()
missing = missing[missing > 0]
if not missing.empty:
    display(Markdown("**Valores ausentes por coluna:**"))
    display(missing.to_frame("Nulos").style.background_gradient(cmap="Reds"))
else:
    display(Markdown("✅ **Sem valores ausentes nas colunas principais.**"))

---
## 6. Preparação das Séries de Mortalidade

Agregação dos dados brutos em duas séries principais:
- **Por faixa etária:** total acumulado no período por grupo etário.
- **Por ano:** total anual de mortes.

In [ ]:
ano_ini_m, ano_fim_m = map(int, ANOS_RANGE_MORTALIDADE)
anos_m = list(range(ano_ini_m, ano_fim_m + 1))

# ── Soma por faixa etária ─────────────────────────────────────────────────────
faixas_por_linha = pd.DataFrame(index=f_mort.index)
for faixa in FAIXAS_ETARIAS_MORTALIDADE:
    cols_faixa = [
        _col_ano_faixa(ano, faixa)
        for ano in anos_m
        if _col_ano_faixa(ano, faixa) in f_mort.columns
    ]
    faixas_por_linha[faixa] = _sum_row_wise(f_mort, cols_faixa)

por_faixa_mort = faixas_por_linha[FAIXAS_ETARIAS_MORTALIDADE].sum(axis=0).astype(int)

# ── Soma por ano ──────────────────────────────────────────────────────────────
por_ano_mort = {}
for ano in anos_m:
    col_total = _col_total_ano(ano)
    if col_total in f_mort.columns:
        total_ano = pd.to_numeric(f_mort[col_total], errors="coerce").fillna(0).sum()
    else:
        cols_ano = [
            _col_ano_faixa(ano, faixa)
            for faixa in FAIXAS_ETARIAS_MORTALIDADE
            if _col_ano_faixa(ano, faixa) in f_mort.columns
        ]
        total_ano = _sum_row_wise(f_mort, cols_ano).sum()
    por_ano_mort[ano] = int(total_ano)

por_ano_mort_s = pd.Series(por_ano_mort).sort_index()

print("✅ Séries de mortalidade calculadas.")
print(f"   Total de mortes no período : {_fmt_int(por_faixa_mort.sum())}")
print(f"   Faixa mais afetada         : {por_faixa_mort.idxmax()} ({_fmt_int(por_faixa_mort.max())} mortes)")

---
## 7. Análise Exploratória Univariada

### 7.1 Mortalidade acumulada por faixa etária

Tabela com os totais absolutos e percentuais por grupo etário no período completo.

In [ ]:
total_geral = int(por_faixa_mort.sum())

tabela_faixas = pd.DataFrame({
    "Faixa Etária"  : por_faixa_mort.index,
    "Valor Absoluto": por_faixa_mort.values,
    "Percentual (%)" : (por_faixa_mort.values / total_geral * 100).round(3),
})

tabela_faixas = pd.concat([
    tabela_faixas,
    pd.DataFrame({
        "Faixa Etária"  : ["Total"],
        "Valor Absoluto": [total_geral],
        "Percentual (%)" : [100.0],
    }),
], ignore_index=True)

display(Markdown(f"### Mortalidade acumulada por faixa etária ({ano_ini_m}–{ano_fim_m})"))
display(
    tabela_faixas.style
    .format({"Valor Absoluto": "{:,.0f}", "Percentual (%)": "{:.3f}"})
    .bar(subset=["Percentual (%)"], color=COR_BARRAS, vmin=0)
    .set_properties(**{"text-align": "center"})
    .set_table_styles([{"selector": "th", "props": [("text-align", "center")]}])
    .highlight_max(subset=["Valor Absoluto"], color="#d4edda")
    .highlight_min(subset=["Valor Absoluto"], color="#f8d7da")
)

### 7.2 Estatísticas descritivas de mortalidade por município

In [ ]:
# Agrega mortalidade total por município a partir de f_mort
base_num  = _drop_index_cols(f_mort)
col_mun_n = _find_first_col(base_num, {"municipios", "municipio", "cidades", "cidade", "nome_municipio"})

if col_mun_n is None:
    raise KeyError("Não foi possível identificar a coluna de município em f_mort.")

cols_total_ano = [c for c in base_num.columns if re.match(r"^Total_\d{4}$", str(c).strip())]

if cols_total_ano:
    anual = base_num[[col_mun_n] + cols_total_ano].copy()
    anual[cols_total_ano] = anual[cols_total_ano].apply(pd.to_numeric, errors="coerce").fillna(0)
    anual = anual.groupby(col_mun_n, as_index=True)[cols_total_ano].sum()
    anual.columns = [int(re.findall(r"\d{4}", c)[0]) for c in anual.columns]
else:
    cols_ano_faixa = [c for c in base_num.columns if re.match(r"^\d{4}_.+", str(c).strip())]
    if not cols_ano_faixa:
        raise KeyError("Nenhuma coluna anual encontrada em f_mort.")
    anual_raw = base_num[[col_mun_n] + cols_ano_faixa].copy()
    anual_raw[cols_ano_faixa] = anual_raw[cols_ano_faixa].apply(pd.to_numeric, errors="coerce").fillna(0)
    anual_raw = anual_raw.groupby(col_mun_n, as_index=True)[cols_ano_faixa].sum()
    anos_uniq = sorted({int(str(c).split("_")[0]) for c in anual_raw.columns})
    anual = pd.DataFrame(index=anual_raw.index)
    for ano in anos_uniq:
        cols_ano = [c for c in anual_raw.columns if str(c).startswith(f"{ano}_")]
        anual[ano] = anual_raw[cols_ano].sum(axis=1)

anual = anual.sort_index(axis=1)
total_por_mun = anual.sum(axis=1)

media_mortes   = total_por_mun.mean()
mediana_mortes = total_por_mun.median()
desvio_mortes  = total_por_mun.std()
anos_totais    = ano_fim_m - ano_ini_m + 1
media_anual_mg = total_por_mun.sum() / anos_totais

# Tabela de estatísticas descritivas
stats_df = pd.DataFrame({
    "Estatística": [
        "Total de mortes no período",
        "Média de mortes por município (acumulado)",
        "Mediana de mortes por município",
        "Desvio padrão por município",
        "Média anual de mortes (total MG)",
    ],
    "Valor": [
        f"{int(total_por_mun.sum()):,.0f}".replace(",", "."),
        f"{media_mortes:,.2f}".replace(",", "."),
        f"{mediana_mortes:,.2f}".replace(",", "."),
        f"{desvio_mortes:,.2f}".replace(",", "."),
        f"{media_anual_mg:,.2f}".replace(",", "."),
    ]
})

display(Markdown(f"### Estatísticas descritivas — mortalidade por município ({ano_ini_m}–{ano_fim_m})"))
display(
    stats_df.style
    .set_properties(**{"text-align": "left"})
    .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}])
    .hide(axis="index")
)

### 7.3 Top 3 municípios com maior mortalidade acumulada

In [ ]:
top3 = (
    total_por_mun
    .nlargest(3)
    .reset_index()
    .rename(columns={col_mun_n: "Município", 0: "Total de mortes"})
)
top3.columns = ["Município", "Total de mortes"]
top3.index   = top3.index + 1  # ranking começa em 1

display(Markdown("### 🏆 Top 3 — municípios com maior mortalidade acumulada"))
display(
    top3.style
    .format({"Total de mortes": "{:,.0f}"})
    .set_properties(**{"text-align": "center"})
    .set_table_styles([{"selector": "th", "props": [("text-align", "center")]}])
    .background_gradient(subset=["Total de mortes"], cmap="Blues")
)

---
## 8. Visualizações — Gráficos de Mortalidade

Dois gráficos independentes com a mesma estética:
- **Gráfico de barras horizontais:** total acumulado por faixa etária.
- **Gráfico de dispersão:** total anual de mortes por ano.

In [ ]:
def _layout_base(titulo: str, width: int, height: int) -> dict:
    """Retorna dict de layout compartilhado entre os gráficos de mortalidade."""
    return dict(
        title=dict(
            text=titulo,
            font=dict(size=FONTE_TITULO, family="Arial", color="#1a1a2e"),
            x=0.5, xanchor="center",
        ),
        height=height, width=width,
        plot_bgcolor="#f8f9fb", paper_bgcolor="white",
        margin=dict(l=160, r=120, t=100, b=80),
        hovermode="closest",
        legend=dict(
            orientation="h", yanchor="bottom", y=1.02,
            xanchor="center", x=0.5,
            font=dict(size=FONTE_LEGENDA),
            bgcolor="rgba(255,255,255,0.85)",
            bordercolor="#ddd", borderwidth=1,
        ),
        hoverlabel=dict(
            bgcolor="white", bordercolor="#aaa",
            font=dict(size=FONTE_EIXO_TICK, family="Arial"),
        ),
    )


def plot_barras_faixa(
    por_faixa: pd.Series,
    anos_range: tuple,
    titulo: str = TITULO_MORTALIDADE,
    cor: str = COR_BARRAS,
    width: int = 1200,
    height: int = 550,
):
    """Gráfico de barras verticais: faixa etária no eixo X, número de mortes no eixo Y."""
    serie = pd.to_numeric(por_faixa, errors="coerce").fillna(0)
    categorias = [str(f) for f in FAIXAS_ETARIAS_MORTALIDADE]
    valores    = [int(serie.get(f, 0)) for f in FAIXAS_ETARIAS_MORTALIDADE]
    ano_ini, ano_fim = map(int, anos_range)

    hover = [
        f"<b>Faixa etária:</b> {cat}<br><b>Total de mortes:</b> {_fmt_int(val)}<br><i>Período {ano_ini}–{ano_fim}</i>"
        for cat, val in zip(categorias, valores)
    ]

    fig = go.Figure()
    fig.add_trace(go.Bar(
        name=f"Total de mortes por faixa etária ({ano_ini}–{ano_fim})",
        x=categorias,
        y=valores,
        marker=dict(color=cor, opacity=0.85, line=dict(color="white", width=1.2)),
        text=[_fmt_int(v) if v > 0 else "" for v in valores],
        textposition="outside",
        textfont=dict(size=16, color="#222"),
        hovertext=hover,
        hoverinfo="text",
        hovertemplate="%{hovertext}<extra></extra>",
    ))

    y_max  = max(v for v in valores if v > 0) * 1.18
    y_step = 2000

    layout = _layout_base(titulo, width, height)
    layout["xaxis"] = dict(
        title="Faixa etária",
        title_font=dict(size=FONTE_EIXO_TITULO),
        tickfont=dict(size=FONTE_EIXO_TICK),
        tickmode="array",
        tickvals=categorias,
        ticktext=categorias,
        showgrid=False,
        showline=True,
        linecolor="#ccc",
        automargin=True,
    )
    layout["yaxis"] = dict(
        title="Número de mortes",
        title_font=dict(size=FONTE_EIXO_TITULO),
        tickfont=dict(size=FONTE_EIXO_TICK),
        gridcolor="#dde3ed",
        gridwidth=1,
        showline=True,
        linecolor="#ccc",
        zeroline=True,
        zerolinecolor="#bbb",
        range=[0, y_max],
        tickvals=list(range(0, int(y_max) + y_step, y_step)),
        ticktext=[_fmt_int(v) for v in range(0, int(y_max) + y_step, y_step)],
    )
    fig.update_layout(**layout)
    fig.show()
    return fig


def plot_dispersao_anual(
    por_ano: pd.Series,
    anos_range: tuple,
    titulo: str = TITULO_MORTALIDADE,
    cor: str = COR_PONTOS,
    width: int = 1200,
    height: int = 1050,
):
    """Gráfico de dispersão: ANO no eixo Y (todos os anos), NÚMERO DE MORTES no eixo X."""
    serie = pd.to_numeric(por_ano, errors="coerce").fillna(0)
    serie = serie[serie > 0].sort_index()

    if serie.empty:
        raise ValueError("Sem dados > 0 para plotar.")

    anos_lista = serie.index.astype(int).tolist()
    valores    = serie.values.astype(int).tolist()
    ano_ini, ano_fim = map(int, anos_range)

    x_max_val   = max(valores)
    x_max_plot  = int(x_max_val * 1.28)
    x_tick_step = 200 if x_max_val < 2000 else 500

    hover = [
        f"<b>Ano:</b> {ano}<br><b>Total de mortes:</b> {_fmt_int(val)}"
        for ano, val in zip(anos_lista, valores)
    ]

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        name=f"Total de mortes por ano ({ano_ini}–{ano_fim})",
        x=valores,
        y=anos_lista,
        mode="markers+text",
        marker=dict(
            color=cor, size=9, opacity=0.90,
            line=dict(color=cor, width=1),
            symbol="circle",
        ),
        text=[_fmt_int(v) for v in valores],
        textposition="middle right",
        textfont=dict(size=12, color=cor),
        hovertext=hover,
        hoverinfo="text",
        hovertemplate="%{hovertext}<extra></extra>",
    ))

    xticks = list(range(0, x_max_plot + 1, x_tick_step))

    layout = _layout_base(titulo, width, height)
    layout["xaxis"] = dict(
        title="Número de mortes",
        title_font=dict(size=FONTE_EIXO_TITULO),
        tickvals=xticks,
        ticktext=[_fmt_int(v) for v in xticks],
        tickfont=dict(size=FONTE_EIXO_TICK),
        gridcolor="#dde3ed",
        gridwidth=1,
        showline=True,
        linecolor="#ccc",
        zeroline=True,
        zerolinecolor="#bbb",
        range=[0, x_max_plot],
    )
    layout["yaxis"] = dict(
        title="Ano",
        title_font=dict(size=FONTE_EIXO_TITULO),
        tickfont=dict(size=12),
        showgrid=True,
        gridcolor="#dde3ed",
        showline=True,
        linecolor="#ccc",
        tickmode="array",
        tickvals=anos_lista,          # todos os anos
        ticktext=[str(a) for a in anos_lista],
        range=[min(anos_lista) - 1, max(anos_lista) + 1],
        dtick=1,
    )
    fig.update_layout(**layout)
    fig.show()
    return fig


print("✅ Funções de visualização definidas.")

### 8.1 Total acumulado por faixa etária (barras)

In [ ]:
plot_barras_faixa(
    por_faixa=por_faixa_mort.reindex(FAIXAS_ETARIAS_MORTALIDADE),
    anos_range=ANOS_RANGE_MORTALIDADE,
    titulo=f"{TITULO_MORTALIDADE} — Total acumulado por faixa etária",
)

### 8.2 Total de mortes por ano (dispersão)

In [ ]:
plot_dispersao_anual(
    por_ano=por_ano_mort_s,
    anos_range=ANOS_RANGE_MORTALIDADE,
    titulo=f"{TITULO_MORTALIDADE} — Total por ano",
)

---
## 9. Mapas de Minas Gerais

Visualização espacial da mortalidade acumulada (1979–2024) em dois níveis de agregação:
- **Por município:** mapa coroplético colorido por classe de mortalidade.
- **Por regional de saúde:** dissolução dos municípios por regional, com categorias próprias.

A malha geográfica é obtida via `geobr` (IBGE) e integrada aos dados via código IBGE normalizado para 7 dígitos.

### 9.1 Mapa por município

In [ ]:
base_mort = _drop_index_cols(f_mort)

col_cod = _find_first_col(base_mort, {"codigo", "cod", "codigo_municipio", "cod_municipio", "codigo_ibge"})
col_mun = _find_first_col(base_mort, {"municipios", "municipio", "cidades", "cidade", "nome_municipio"})

if col_cod is None:
    raise KeyError(f"Coluna de código não encontrada.\n{base_mort.columns.tolist()}")

# Soma acumulada por município
anos_mapa  = set(range(ANO_INI_MAPA, ANO_FIM_MAPA + 1))
cols_faixa = [
    c for c in base_mort.columns
    if re.match(r"^\d{4}_.+", str(c).strip())
    and int(str(c).split("_", 1)[0]) in anos_mapa
]
if not cols_faixa:
    raise ValueError("Nenhuma coluna de faixa etária (YYYY_faixa) encontrada.")

acumulado_linha = base_mort[cols_faixa].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)

base_acum = pd.DataFrame({
    "codigo_municipio": _norm_cod7(base_mort[col_cod]),
    "nome_municipio"  : base_mort[col_mun].astype(str).str.strip() if col_mun else "Município",
    "mortes_1979_2024": acumulado_linha,
})

por_municipio = (
    base_acum
    .groupby(["codigo_municipio", "nome_municipio"], as_index=False)["mortes_1979_2024"]
    .sum()
)

# Malha geográfica (CRS geográfico 4326 mantido para o plotly)
mg = geobr.read_municipality(code_muni="MG", year=ANO_MALHA_MUNICIPIOS_MG).to_crs(4326)
mg["codigo_municipio"] = _norm_cod7(mg["code_muni"])

# Merge malha + dados
mapa_mun = mg[["codigo_municipio", "geometry"]].merge(por_municipio, on="codigo_municipio", how="left")
mapa_mun["nome_municipio"]   = mapa_mun["nome_municipio"].fillna("")
mapa_mun["mortes_1979_2024"] = pd.to_numeric(mapa_mun["mortes_1979_2024"], errors="coerce").fillna(0)

# Classificação em faixas e cores
classe_mun, ordem_mun = _rotular_faixas_fixas(mapa_mun["mortes_1979_2024"], FAIXAS_MUNICIPIOS)
mapa_mun["classe_mortes"] = classe_mun

cores_mun = {"Sem mortalidade (0)": COR_SEM_MORTALIDADE}
for i, lab in enumerate(reversed([x for x in ordem_mun if x != "Sem mortalidade (0)"])):
    cores_mun[lab] = PALETA_MAPA[min(i, len(PALETA_MAPA) - 1)]

geojson_mun = json.loads(mapa_mun[["codigo_municipio", "geometry"]].to_json())

sem_match = (mapa_mun["mortes_1979_2024"] == 0).sum()
print(f"Municípios na malha geobr:           {len(mg)}")
print(f"Municípios na tabela de mortalidade: {len(por_municipio)}")
print(f"Municípios sem match (mortes = 0):   {sem_match}")

fig_mun = px.choropleth(
    mapa_mun,
    geojson=geojson_mun,
    locations="codigo_municipio",
    featureidkey="properties.codigo_municipio",
    color="classe_mortes",
    color_discrete_map=cores_mun,
    category_orders={"classe_mortes": ordem_mun},
    hover_name="nome_municipio",
    hover_data={"codigo_municipio": True, "mortes_1979_2024": ":,.0f", "classe_mortes": True},
    labels={"classe_mortes": "Mortalidade (1979–2024)", "mortes_1979_2024": "Total de mortes"},
    title="Mortalidade acumulada por município (1979–2024)",
)
fig_mun.update_geos(
    fitbounds="locations", visible=False,
    showcountries=False, showcoastlines=False, showland=False, showframe=False,
)
fig_mun.update_traces(marker_line_color="#2d3436", marker_line_width=0.8)
fig_mun.update_layout(
    template="plotly_white",
    width=1250, height=820,
    margin=dict(l=10, r=10, t=80, b=10),
    title=dict(x=0.5, xanchor="center", font=dict(size=20)),
    legend=dict(orientation="h", yanchor="top", y=0.99,
                xanchor="center", x=0.5, title=dict(text="")),
)
fig_mun.show()
print("✅ Mapa por município gerado.")

### 9.2 Mapa por Regional de Saúde

Os municípios são agrupados por regional de saúde usando a tabela auxiliar `tabela_municipios_regionais.csv`. Municípios sem regional associada recebem a regional do vizinho mais próximo (preenchimento por proximidade geoespacial com CRS projetado SIRGAS 2000 / UTM 23S).

In [ ]:
# Localização automática da tabela de municípios × regionais
candidatos_base = [Path.cwd().resolve()] + list(Path.cwd().resolve().parents)
path_municipios = None
for base in candidatos_base:
    pm = base / "Tratamento" / "Municípios" / "tabela_municipios_regionais.csv"
    if pm.exists():
        path_municipios = pm
        break

if path_municipios is None:
    raise FileNotFoundError(
        "Não foi possível localizar 'tabela_municipios_regionais.csv'. "
        "Verifique se o arquivo existe em <raiz>/Tratamento/Municípios/."
    )

df_mun_reg = pd.read_csv(path_municipios)
df_mun_reg.columns              = [c.strip() for c in df_mun_reg.columns]
df_mun_reg["codigo_ibge"]       = _norm_cod7(df_mun_reg["codigo_ibge"])
df_mun_reg["regional_original"] = df_mun_reg["regional_original"].fillna("Nenhum")

# Join mortalidade × regional → soma por regional
base_reg = base_acum[["codigo_municipio", "mortes_1979_2024"]].rename(
    columns={"mortes_1979_2024": "mortes"}
).merge(
    df_mun_reg[["codigo_ibge", "regional_original"]],
    left_on="codigo_municipio", right_on="codigo_ibge", how="left",
)
base_reg["regional_original"] = base_reg["regional_original"].fillna("Nenhum")

por_regional = (
    base_reg[base_reg["regional_original"] != "Nenhum"]
    .groupby("regional_original", as_index=False)["mortes"]
    .sum()
)

# Categorias e paleta para regionais
ordem_reg = [
    f"0–{FAIXAS_REGIONAIS[0]}",
    *[f"{FAIXAS_REGIONAIS[i]+1:,}–{FAIXAS_REGIONAIS[i+1]:,}".replace(",", ".") for i in range(len(FAIXAS_REGIONAIS)-1)],
    f"{FAIXAS_REGIONAIS[-1]+1:,}+".replace(",", "."),
]
cores_reg = dict(zip(reversed(ordem_reg), PALETA_MAPA))
por_regional["categoria_mortes"] = por_regional["mortes"].apply(
    lambda n: _categoria_reg(n, FAIXAS_REGIONAIS)
)

# Dissolve municípios por regional
gdf_mg_reg = mg[["codigo_municipio", "geometry"]].merge(
    df_mun_reg[["codigo_ibge", "regional_original"]],
    left_on="codigo_municipio", right_on="codigo_ibge", how="left",
)
gdf_mg_reg["regional_original"] = gdf_mg_reg["regional_original"].fillna("Nenhum")

# Preenchimento geoespacial com CRS projetado (evita UserWarning de distância em graus)
sem_reg_mask = gdf_mg_reg["regional_original"] == "Nenhum"
if sem_reg_mask.any():
    # Reprojetar para SIRGAS 2000 / UTM zona 23S (EPSG:31983) — distância em metros
    gdf_proj      = gdf_mg_reg.to_crs(epsg=31983)
    com_regional  = gdf_proj[~sem_reg_mask].copy()
    sem_regional  = gdf_proj[sem_reg_mask].index
    for idx in sem_regional:
        geom = gdf_proj.loc[idx, "geometry"]
        distancias = com_regional.geometry.distance(geom)
        gdf_mg_reg.loc[idx, "regional_original"] = com_regional.loc[distancias.idxmin(), "regional_original"]

gdf_regionais = (
    gdf_mg_reg[["regional_original", "geometry"]]
    .dissolve(by="regional_original", as_index=False)
)

gdf_regionais = gdf_regionais.merge(por_regional, on="regional_original", how="left")
gdf_regionais["mortes"]           = gdf_regionais["mortes"].fillna(0).astype(int)
gdf_regionais["categoria_mortes"] = gdf_regionais["categoria_mortes"].fillna(ordem_reg[0])
gdf_regionais["categoria_mortes"] = pd.Categorical(
    gdf_regionais["categoria_mortes"], categories=ordem_reg, ordered=True
)

gdf_regionais_plot       = gdf_regionais.reset_index(drop=True).copy()
gdf_regionais_plot["id"] = gdf_regionais_plot.index.astype(str)

lat_c, lon_c = _centro_gdf(gdf_regionais_plot)

fig_reg = px.choropleth_map(
    gdf_regionais_plot,
    geojson=json.loads(gdf_regionais_plot.to_json()),
    locations="id",
    color="categoria_mortes",
    color_discrete_map=cores_reg,
    category_orders={"categoria_mortes": ordem_reg},
    title=f"Mortalidade acumulada por Regional de Saúde ({ANO_INI_MAPA}–{ANO_FIM_MAPA})",
    hover_name="regional_original",
    hover_data={"mortes": True, "categoria_mortes": False, "id": False},
    labels={"mortes": "Total de mortes"},
    map_style="white-bg",
    center={"lat": lat_c, "lon": lon_c},
    zoom=5.5, opacity=0.95,
)
fig_reg.update_layout(
    margin=dict(l=10, r=10, t=60, b=10),
    title_x=0.5, height=760,
    legend_title_text="Mortes",
)
fig_reg.show()
print("✅ Mapa por regional gerado.")

---
## 10. Conclusões da EDA — Mortalidade

Síntese dos principais achados desta etapa exploratória.

### Distribuição por faixa etária
- A mortalidade por câncer de mama concentra-se fortemente nas faixas acima de 40 anos, com pico na faixa **70 ou mais**.
- As faixas **0–9** e **10–19** apresentam valores próximos de zero, condizente com a literatura epidemiológica.

### Tendência temporal
- A série anual evidencia tendência de crescimento ao longo do período 1979–2024, tanto no total absoluto quanto nos blocos quinquenais.

### Distribuição espacial
- Os municípios da região metropolitana de Belo Horizonte e do Triângulo Mineiro concentram os maiores totais absolutos.
- Municípios menores e regionais do norte/nordeste de MG apresentam mortalidade acumulada mais baixa, possivelmente associada a subregistro ou menor densidade populacional.

> **Próxima etapa (PFC II):** Aplicação de K-means clustering sobre os municípios de MG e análise bivariada entre mortalidade e variáveis socioeconômicas (IDH, renda, acesso a serviços de saúde).